# Google Books AI Assistant — Multi-Tool Calling with SQLite

## 1. Imports & Core Dependencies

In [ ]:
import os
import json
import sqlite3
import requests
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr


## 2. Environment Setup & API Client Initialization

In [ ]:
load_dotenv(override=True)

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
gemini_api_key = os.getenv("GEMINI_API_KEY")
google_books_key = os.getenv("Google_Books_Key")

openrouter_url = "https://openrouter.ai/api/v1"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)

gemini_model = "gemini-3.6-flash"
openrouter_model = "gemini-2.5-flash"


## 3. SQLite Database Setup & Schema Creation

In [ ]:
DB = "books.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS saved_books (
        volume_id TEXT PRIMARY KEY,
        title TEXT,
        authors TEXT,
        publisher TEXT,
        published_date TEXT,
        synopsis TEXT,
        preview_link TEXT
    )
    ''')
conn.commit()


## 4. System Prompt & Assistant Persona

In [ ]:
SYSTEM_MESSAGE = """
You are a helpful AI Google Books Assistant with local SQLite database integration.
- Use `search_google_books` to search live books from Google Books API.
- Use `save_book_to_db` when the user asks to save or bookmark a book. If the user says 'save it', 'save this one', or 'save all', call save_book_to_db for each book they want saved. Do not ask for reconfirmation.
- Use `get_saved_books` when the user asks to see their saved/bookmarked books.
Always give clear, friendly responses with titles, synopses, and preview links.
"""


## 5. Tool Functions — Search, Save & Retrieve

In [ ]:
BASE_URL = "https://www.googleapis.com/books/v1/volumes"
def search_google_books(query):
    """Searches live books via Google Books API without auto-saving."""
    print(f"🔍 [TOOL CALLED] search_google_books(query='{query}')", flush=True)
    params = {
        "q": query,
        "maxResults": 5,
        "key": google_books_key
    }
    res = requests.get(BASE_URL, params=params)
    if res.status_code != 200:
        return f"Error fetching books: {res.text}"
    
    data = res.json()
    items = data.get("items", [])
    if not items:
        return f"No books found for query: '{query}'"
    
    results = []
    for item in items:
        vol = item.get("volumeInfo", {})
        synopsis = vol.get("description", "No synopsis available.")
        results.append({
            "volume_id": item.get("id"),
            "title": vol.get("title", "Unknown"),
            "authors": ", ".join(vol.get("authors", ["Unknown"])),
            "publisher": vol.get("publisher", "N/A"),
            "published_date": vol.get("publishedDate", "N/A"),
            "synopsis": synopsis[:200] + "..." if len(synopsis) > 200 else synopsis,
            "preview_link": vol.get("previewLink", "N/A")
        })
        print(f"   -> Found {len(results)} books from Google Books API.", flush=True)

    return json.dumps(results, indent=2)

In [ ]:
def save_book_to_db(volume_id, title, authors, publisher="N/A", published_date="N/A", synopsis="N/A", preview_link="N/A"):
    """Saves a specific book to the local SQLite database when requested by the user."""
    print(f"💾 [TOOL CALLED] save_book_to_db(title='{title}', volume_id='{volume_id}')", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('''
        INSERT OR REPLACE INTO saved_books (volume_id, title, authors, publisher, published_date, synopsis, preview_link)
        VALUES (?, ?, ?, ?, ?, ?, ?)
        ''', (volume_id, title, authors, publisher, published_date, synopsis, preview_link))
        conn.commit()
        print(f"   -> Saved '{title}' to SQLite database ('books.db').", flush=True)
    return f"Successfully saved '{title}' (ID: {volume_id}) to your SQLite database!"

In [ ]:
def get_saved_books():
    """Retrieves all saved books from SQLite database."""
    print(f"📁 [TOOL CALLED] get_saved_books() - querying SQLite database 'books.db'", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT volume_id, title, authors, publisher, published_date, synopsis, preview_link FROM saved_books')
        rows = cursor.fetchall()
        if not rows:
            return "No books currently saved in your SQLite database."
        
        books = []
        for r in rows:
            books.append({
                "volume_id": r[0],
                "title": r[1],
                "authors": r[2],
                "publisher": r[3],
                "published_date": r[4],
                "synopsis": r[5][:200] + "..." if r[5] and len(r[5]) > 200 else r[5],
                "preview_link": r[6]
            })
        return json.dumps(books, indent=2)

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_google_books",
            "description": "Search live Google Books API by topic, title, or author.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search term e.g. 'python programming' or 'inauthor:steve jobs'"
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_book_to_db",
            "description": "Save a specific book into the local SQLite database when the user explicitly requests to save/bookmark it.",
            "parameters": {
                "type": "object",
                "properties": {
                    "volume_id": {"type": "string"},
                    "title": {"type": "string"},
                    "authors": {"type": "string"},
                    "publisher": {"type": "string"},
                    "published_date": {"type": "string"},
                    "synopsis": {"type": "string"},
                    "preview_link": {"type": "string"}
                },
                "required": ["volume_id", "title", "authors"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_saved_books",
            "description": "Retrieve all saved/bookmarked books from the local SQLite database.",
            "parameters": {
                "type": "object",
                "properties": {},
                "additionalProperties": False
            }
        }
    }
]

## 7. Tool Execution Handler

In [ ]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        func_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        print(f"\n⚙️ [LLM TRIGGERED TOOL] Function: {func_name} | Arguments: {args}", flush=True)

        if func_name == "search_google_books":
            result = search_google_books(args.get("query"))
        elif func_name == "save_book_to_db":
            result = save_book_to_db(
                volume_id=args.get("volume_id"),
                title=args.get("title"),
                authors=args.get("authors"),
                publisher=args.get("publisher", "N/A"),
                published_date=args.get("published_date", "N/A"),
                synopsis=args.get("synopsis", "N/A"),
                preview_link=args.get("preview_link", "N/A")
            )
        elif func_name == "get_saved_books":
            result = get_saved_books()
        else:
            result = "Unknown tool call"
            
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    return responses


## 8. Main Chat Loop — Agentic Multi-Turn Engine

In [ ]:
def chat(message, history):
    formatted_history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": SYSTEM_MESSAGE}] + formatted_history + [{"role": "user", "content": message}]
    
    while True:
        response = openrouter.chat.completions.create(model=openrouter_model, messages=messages, tools=tools)
        
        if not response.choices:
            return "API returned an empty response. Please try again."
        
        choice = response.choices[0]
        
        if choice.message.tool_calls:
            assistant_msg = choice.message
            tool_responses = handle_tool_calls(assistant_msg)
            messages.append(assistant_msg)
            messages.extend(tool_responses)
        else:
            return choice.message.content


## 9. User Interface — Gradio Chat UI Launch

In [ ]:
gr.ChatInterface(fn=chat, title="Google Books AI Assistant").launch()
